In [2]:
import matplotlib.pyplot as plt
from fpdf import FPDF
import os

# --- CONFIGURAÇÃO DE DADOS REAIS (CAMETÁ) ---
# Apenas para exemplificar valores numéricos no PDF se necessário
R_TERRA = 6371000

class ScientificPDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.cell(0, 10, 'Relatório Técnico: Modelagem Matemática Geoespacial', 0, 1, 'C')
        self.set_font('Arial', 'I', 10)
        self.cell(0, 5, 'Aplicação: Projeção Ortogonal em Coordenadas Geodésicas', 0, 1, 'C')
        self.ln(10)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'C')

    def section_title(self, title):
        self.set_font('Arial', 'B', 12)
        self.set_fill_color(230, 230, 230)
        self.cell(0, 8, title, 0, 1, 'L', 1)
        self.ln(4)

    def body_text(self, text):
        self.set_font('Times', '', 12)
        self.multi_cell(0, 6, text)
        self.ln(2)

def render_latex(formula, filename, fontsize=16):
    """
    Renderiza uma string LaTeX em uma imagem PNG transparente usando Matplotlib.
    """
    fig = plt.figure(figsize=(0.1, 0.1)) # Tamanho dummy, será redimensionado
    fig.text(0, 0, f"${formula}$", fontsize=fontsize)
    
    # Salva temporariamente para calcular o bbox (caixa delimitadora) correta
    output_path = filename
    
    # Configurações para remover eixos e fundo
    ax = plt.gca()
    ax.axis('off')
    
    # Salva a imagem recortada exatamente no tamanho da fórmula
    plt.savefig(output_path, dpi=300, bbox_inches='tight', transparent=True, pad_inches=0.05)
    plt.close()

def generate_scientific_report():
    pdf = ScientificPDF()
    pdf.add_page()

    # --- INTRODUÇÃO ---
    pdf.section_title("1. Definição do Problema")
    pdf.body_text(
        "O objetivo é determinar as coordenadas geográficas de um ponto $P$ que representa a projeção "
        "ortogonal de um vértice $B$ sobre um segmento geodésico definido por $A$ e $C$. "
        "Dada a pequena magnitude das distâncias envolvidas na região de Cametá (< 1 km), "
        "adota-se a aproximação do plano tangente local, negligenciando a curvatura esférica "
        "para fins de simplificação vetorial Euclidiana."
    )

    # --- FASE 1 ---
    pdf.section_title("2. Linearização (Transformação Geodésica)")
    pdf.body_text(
        "Sejam as coordenadas geográficas (latitude $\phi$, longitude $\lambda$). "
        "A conversão para um sistema cartesiano local $(x, y)$ em metros, com origem em $A$, é dada por:"
    )

    # Equação 1: Fatores de conversão
    eq1 = r"k_{\phi} = \frac{2 \pi R}{360}, \quad k_{\lambda} = k_{\phi} \cdot \cos(\phi_m)"
    render_latex(eq1, "eq1.png")
    pdf.image("eq1.png", w=80, x=65) # Centralizado
    
    pdf.body_text("Onde $R$ é o raio da Terra e $\phi_m$ é a latitude média. As coordenadas cartesianas locais são obtidas por:")
    
    # Equação 2: Transformação Linear
    eq2 = r"\begin{cases} x = (\lambda - \lambda_A) \cdot k_{\lambda} \\ y = (\phi - \phi_A) \cdot k_{\phi} \end{cases}"
    render_latex(eq2, "eq2.png")
    pdf.image("eq2.png", w=70, x=70)
    
    pdf.ln(5)

    # --- FASE 2 ---
    pdf.section_title("3. Álgebra Vetorial no Espaço Euclidiano")
    pdf.body_text(
        "No espaço métrico linearizado $\mathbb{R}^2$, definimos os vetores de deslocamento a partir da origem $A$:"
    )

    # Equação 3: Vetores
    eq3 = r"\vec{u} = \vec{AC} = (x_C, y_C), \quad \vec{v} = \vec{AB} = (x_B, y_B)"
    render_latex(eq3, "eq3.png")
    pdf.image("eq3.png", w=80, x=65)

    pdf.body_text(
        "A projeção ortogonal do vetor $\vec{v}$ sobre a reta suporte definida por $\vec{u}$ é obtida "
        "pelo cálculo do escalar de projeção $t$, derivado do produto interno (dot product):"
    )

    # Equação 4: Fator t
    eq4 = r"t = \frac{\vec{u} \cdot \vec{v}}{\| \vec{u} \|^2} = \frac{x_C x_B + y_C y_B}{x_C^2 + y_C^2}"
    render_latex(eq4, "eq4.png")
    pdf.image("eq4.png", w=80, x=65)

    pdf.body_text(
        "O ponto projetado $P$ no sistema local é determinado pela combinação linear:"
    )

    # Equação 5: Ponto P local
    eq5 = r"P_{(x,y)} = A_{(0,0)} + t \cdot \vec{u} \implies (x_P, y_P) = (t \cdot x_C, \,\, t \cdot y_C)"
    render_latex(eq5, "eq5.png")
    pdf.image("eq5.png", w=100, x=55)

    pdf.ln(5)

    # --- FASE 3 ---
    pdf.section_title("4. Deslinearização (Retorno às Coordenadas Globais)")
    pdf.body_text(
        "Finalmente, as coordenadas locais $(x_P, y_P)$ são convertidas de volta para o sistema geodésico "
        "invertendo a transformação da Fase 1:"
    )

    # Equação 6: Final
    eq6 = r"\begin{cases} \phi_P = \phi_A + \frac{y_P}{k_{\phi}} \\ \lambda_P = \lambda_A + \frac{x_P}{k_{\lambda}} \end{cases}"
    render_latex(eq6, "eq6.png")
    pdf.image("eq6.png", w=60, x=75)

    pdf.ln(10)
    pdf.set_font('Arial', 'I', 10)
    pdf.multi_cell(0, 5, "Nota: Este modelo assume isometria local, válida para distâncias pequenas onde a distorção esférica é desprezível.")

    # --- GERAR PDF ---
    output_filename = "Relatorio_Modelagem_Cientifica.pdf"
    pdf.output(output_filename)
    
    # Limpeza dos arquivos temporários
    temp_files = ["eq1.png", "eq2.png", "eq3.png", "eq4.png", "eq5.png", "eq6.png"]
    for f in temp_files:
        if os.path.exists(f):
            os.remove(f)

    print(f"Sucesso! O arquivo '{output_filename}' foi gerado com notação LaTeX.")

if __name__ == "__main__":
    try:
        generate_scientific_report()
    except Exception as e:
        print(f"Erro: {e}")

Erro: 
\begin{cases} x = (\lambda - \lambda_A) \cdot k_{\lambda} \\ y = (\phi - \phi_A) \cdot k_{\phi} \end{cases}
^
ParseFatalException: Unknown symbol: \begin, found '\'  (at char 0), (line:1, col:1)
Error in callback <function _draw_all_if_interactive at 0x000001E9BA24BEB0> (for post_execute), with arguments args (),kwargs {}:


ValueError: 
\begin{cases} x = (\lambda - \lambda_A) \cdot k_{\lambda} \\ y = (\phi - \phi_A) \cdot k_{\phi} \end{cases}
^
ParseFatalException: Unknown symbol: \begin, found '\'  (at char 0), (line:1, col:1)

ValueError: 
\begin{cases} x = (\lambda - \lambda_A) \cdot k_{\lambda} \\ y = (\phi - \phi_A) \cdot k_{\phi} \end{cases}
^
ParseFatalException: Unknown symbol: \begin, found '\'  (at char 0), (line:1, col:1)

<Figure size 10x10 with 1 Axes>